Use this notebook to prepare input data for de novo cases (json file and structures) --NanoDesigner

Skip until part two the notebook if the epitope informations is to be extracted from another nanobody-antigen or antibody-antigen PDB complex.


In [ ]:
# ============================================================================
#  SETUP - run this first
# ============================================================================
import os
import shutil
import sys


def _find_project_dir(start=None):
    """Walk up from the working directory looking for the NanoDesigner clone.

    A notebook does not know its own path, so the repository root is found by
    the directories every clone contains rather than by a relative path.
    """
    markers = ("dyMEAN", "ADesigner", "config_files")
    here = os.path.abspath(start or os.getcwd())
    candidates = [here] + [
        os.path.abspath(os.path.join(here, *([os.pardir] * n))) for n in range(1, 5)
    ]
    for candidate in candidates:
        if all(os.path.isdir(os.path.join(candidate, m)) for m in markers):
            return candidate
    return None


PROJECT_DIR = _find_project_dir()

# If detection fails, or you have more than one clone and it picked the wrong
# one, set the path by hand here instead:
# PROJECT_DIR = "/path/to/NanoDesigner"

if PROJECT_DIR is None:
    raise RuntimeError(
        "Could not locate the NanoDesigner clone starting from " + os.getcwd() +
        ".\nSet PROJECT_DIR manually in this cell."
    )

sys.path.insert(0, PROJECT_DIR)

# Required by the dr_sasa lookup used for epitope extraction in Part 2.
os.environ["NANOBODIES_PROJECT_DIR"] = PROJECT_DIR

# Where this notebook writes the JSON and structures it produces.
WORK_DIR = os.path.join(PROJECT_DIR, "NanoDesigner_test_Cases", "denovo_epitope_info")

# Optional local copy of the SAbDab archive, created by
# Data_download_and_processing/scripts/download_and_process_data.py.
# It is only a cache: structures absent from it are downloaded on demand.
ALL_STRUCTURES_DIR = os.path.join(PROJECT_DIR, "Data_download_and_processing", "all_structures")

_dr_sasa = os.path.join(PROJECT_DIR, "dr_sasa_n", "build", "dr_sasa")

print("PROJECT_DIR       :", PROJECT_DIR)
print("WORK_DIR          :", WORK_DIR)
print("ALL_STRUCTURES_DIR:", ALL_STRUCTURES_DIR,
      "(found)" if os.path.isdir(ALL_STRUCTURES_DIR) else "(absent - will download on demand)")
print("dr_sasa           :", _dr_sasa,
      "(found)" if os.path.isfile(_dr_sasa) else "(MISSING - Part 2 epitope extraction will fail)")


In [ ]:
# use nanodesigner1 kernel
import sys
import os
import shutil
import json
import traceback
# PROJECT_DIR is already on sys.path (see setup cell)
from functionalities.complex_analysis import (
    fetch_from_sabdab, fetch_from_pdb, extract_antibody_info,
    get_cdr_pos, extract_seq_info_from_pdb
)
from dyMEAN.data.pdb_utils import Protein  
from dyMEAN.utils.renumber import renumber_pdb
from functionalities.nanobody_antibody_interacting_residues import interacting_residues, interacting_residues_extended


In [ ]:
# ============================================================================
#  Structure download
# ============================================================================
import shutil

from functionalities.complex_analysis import fetch_from_sabdab, fetch_from_pdb


def _is_structure(path):
    """True only if the file really holds coordinates."""
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        return False
    with open(path, errors="ignore") as fh:
        head = fh.read(400).lower()
    if "<html" in head or "<!doctype" in head:
        return False  # an error page, not a structure
    with open(path, errors="ignore") as fh:
        return any(line.startswith(("ATOM", "HETATM")) for line in fh)


def fetch_structure(pdb_id, dest, numbering="imgt", local_dir=None):
    """Fetch a structure, trying the local cache, then SAbDab, then RCSB.

    Each source is verified before it is accepted, so a failed download raises
    here instead of producing an empty JSON much later.
    """
    tried = []

    if local_dir:
        cached = os.path.join(local_dir, numbering, pdb_id + ".pdb")
        if _is_structure(cached):
            shutil.copy(cached, dest)
            print(f"  {pdb_id}: local copy ({cached})")
            return dest
        tried.append("local cache")

    for name, call in (
        ("SAbDab", lambda: fetch_from_sabdab(pdb_id, numbering, dest, tries=3)),
        ("RCSB", lambda: fetch_from_pdb(pdb_id, dest, tries=3)),
    ):
        try:
            call()
        except Exception as exc:
            print(f"  {pdb_id}: {name} raised {exc!r}")
        if _is_structure(dest):
            print(f"  {pdb_id}: downloaded from {name}")
            return dest
        if os.path.exists(dest):
            os.remove(dest)  # discard the error page before the next attempt
        tried.append(name)

    raise FileNotFoundError(
        f"No usable structure for '{pdb_id}'. Tried: {', '.join(tried)}. "
        f"Check the PDB code, or put the file at "
        f"{os.path.join(local_dir or '<all_structures>', numbering, pdb_id + '.pdb')}."
    )


print("fetch_structure ready (local cache -> SAbDab -> RCSB, each verified)")


In [ ]:
# Fill in this dictionary with the required information:
config = {
    "Nano_source_pdb": "6lr7",
    "nano_chain_id": "B", #heavy chain
    "light_chain_id": "", # for nanobodies, define empty string ""
    "Antigen_source_pdb": "4obe",
    "antigen_chain_id":["A"],
    "epitope_sequences": ["NHFVDEYDPTIEDSYR","TAGQEEYSAMRDQYMRTGE", "YKLV;CLL"],  # defined by the user
    "out_dir": WORK_DIR,
    "numbering_scheme" : "imgt", # imgt or chothia
  }


In [ ]:
# Set the output directory and numbering scheme
out_dir = config["out_dir"]
os.makedirs(out_dir, exist_ok=True)

# Create a temporary directory within the output directory
tmp_dir = os.path.join(out_dir, "temporal_files")
os.makedirs(tmp_dir, exist_ok=True)

# Define the directory to look for pre-existing structures
all_structures_dir = ALL_STRUCTURES_DIR
numbering_scheme = config["numbering_scheme"]

print("Output directory created:", out_dir)
print("Temporary directory created:", tmp_dir)

In [ ]:
# Download the scaffold nanobody and the target antigen.
nano = config["Nano_source_pdb"]
nano_id = config["nano_chain_id"]
antigen = config["Antigen_source_pdb"]
antigen_id = config["antigen_chain_id"]
numbering_scheme = config["numbering_scheme"]

nano_source = os.path.join(tmp_dir, nano + "_raw.pdb")
antigen_source = os.path.join(tmp_dir, antigen + "_raw.pdb")

print("Scaffold:")
fetch_structure(nano, nano_source, numbering_scheme, all_structures_dir)
print("Antigen:")
fetch_structure(antigen, antigen_source, numbering_scheme, all_structures_dir)


In [ ]:
# Initialize JSON content with basic information
json_content = {
    "pdb": nano + f"_{antigen}",
    "heavy_chain": nano_id,
    "light_chain": config["light_chain_id"],
    "antigen_chains": antigen_id
}

# Path for cleaned antigen PDB file
antigen_ = os.path.join(tmp_dir, antigen + ".pdb")
json_content["antigen_source"] = antigen_

# Extract antigen sequence
try:
    pdb = Protein.from_pdb(antigen_source, antigen_id)  # Assume antigen_id is a list
    json_content['antigen_seqs'] = []

    for peptide_id, peptide in pdb.peptides.items():
        sequence = peptide.get_seq()
        print(f"Antigen Peptide ID: {peptide_id}, Sequence: {sequence}")
        json_content['antigen_seqs'].append(sequence)

    # Save the cleaned antigen structure
    pdb.to_pdb(antigen_)
except Exception as e:
    print(f"Error processing antigen sequence: {e}")


In [ ]:
# Define cleaned nanobody PDB path
nano_ = os.path.join(tmp_dir, nano + ".pdb")
json_content["nano_source"] = nano_

# Renumber the nanobody and extract its sequence
try:
    renumber_pdb(nano_source, nano_, numbering_scheme)
    pdb = Protein.from_pdb(nano_, [nano_id])  # nano_id should be a list
    json_content['heavy_chain_seq'] = ""

    for peptide_id, peptide in pdb.peptides.items():
        sequence = peptide.get_seq()
        print(f"Nanobody Peptide ID: {peptide_id}, Sequence: {sequence}")
        json_content['heavy_chain_seq'] += sequence

    # Extract CDR positions and sequences
    cdr_pos_dict = extract_antibody_info(pdb, nano_id, "", numbering_scheme)
    nano_peptide = pdb.peptides.get(nano_id)

    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()
        cdr_pos = get_cdr_pos(cdr_pos_dict, cdr_name)
        json_content[f'cdr{cdr_name}_pos'] = cdr_pos
        start, end = cdr_pos
        cdr_seq = nano_peptide.get_span(start, end + 1).get_seq()
        json_content[f'cdr{cdr_name}_seq'] = cdr_seq
except Exception as e:
    print(f"Error processing nanobody sequence: {e}")


In [ ]:
# Define the output directory for the nanobody-antigen combination
nanobody_id = config["Nano_source_pdb"]
antigen_id = config["Antigen_source_pdb"]
output_dir = os.path.join(out_dir, f"{nanobody_id}_{antigen_id}")
os.makedirs(output_dir, exist_ok=True)
print(f"Saving JSON files to directory: {output_dir}")

# Prepare paths
nano_source = os.path.join(tmp_dir, f"{nanobody_id}.pdb")
antigen_source = os.path.join(tmp_dir, f"{antigen_id}.pdb")
numbering_scheme = config["numbering_scheme"]

# Initialize JSON content for shared fields
json_content = {
    "pdb": f"{nanobody_id}_{antigen_id}",
    "heavy_chain": config["nano_chain_id"],
    "light_chain": config["light_chain_id"],
    "antigen_chains": config["antigen_chain_id"],
    "antigen_source": antigen_source,
    "nano_source": nano_source
}

try:
    ag_chains_to_reconstruct = config["antigen_chain_id"]
    pdb = Protein.from_pdb(antigen_source, ag_chains_to_reconstruct)
    json_content['antigen_seqs'] = [peptide.get_seq() for peptide_id, peptide in pdb.peptides.items()]
    
    # Save cleaned antigen structure
    pdb.to_pdb(antigen_source)
except Exception as e:
    print(f"Error processing antigen sequence: {e}")
    print(traceback.format_exc())

try:
    renumber_pdb(nano_source, nano_source, numbering_scheme)    
    pdb = Protein.from_pdb(nano_source, [config["nano_chain_id"]])
    json_content['heavy_chain_seq'] = "".join(peptide.get_seq() for peptide_id, peptide in pdb.peptides.items())
    
    # Extract CDR positions
    cdr_pos_dict = extract_antibody_info(pdb, config["nano_chain_id"], config["light_chain_id"], numbering_scheme)
    
    # Get CDR sequences using CDR positions
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., "h1", "h2", "h3"
        cdr_key = f"cdr{cdr_name}_pos"
        cdr_seq_key = f"cdr{cdr_name}_seq"
        
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        if cdr_pos:
            json_content[cdr_key] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb.peptides.get(config["nano_chain_id"])
            json_content[cdr_seq_key] = nano_peptide.get_span(start, end + 1).get_seq()
except Exception as e:
    print(f"Error processing nanobody sequence and CDR extraction: {e}")
    print(traceback.format_exc())

list_of_dicts = []
for count, epitope in enumerate(config["epitope_sequences"], start=1):
    function_results = []

    # Split epitope sequences if there are multiple segments
    if ";" in epitope:
        epitope_list = epitope.split(";")
        for i, substring in enumerate(epitope_list, start=1):
            epitope_pdb = extract_seq_info_from_pdb(antigen_source, config["antigen_chain_id"][0], substring)
            if epitope_pdb is not None:
                function_results.extend(epitope_pdb)
            else:
                print(f"Warning: Could not extract information for epitope segment '{substring}' in epitope '{epitope}'")
    else:
        epitope_pdb = extract_seq_info_from_pdb(antigen_source, config["antigen_chain_id"][0], epitope)
        if epitope_pdb is not None:
            function_results.extend(epitope_pdb)
        else:
            print(f"Warning: Could not extract information for epitope '{epitope}'")

    # Create a new dictionary with the epitope information
    new_dict = json_content.copy()
    new_dict["numbering"] = numbering_scheme
    new_dict["pdb"] = f"{nanobody_id}_{antigen_id}_ep_{count}"
    new_dict["epitope_user_input"] = "yes"
    new_dict['epitope'] = function_results  # Add the extracted epitope data
    list_of_dicts.append(new_dict)

    # Save each epitope dictionary as a separate JSON file in the output directory
    epitope_filename = os.path.join(output_dir, f"{new_dict['pdb']}.json")
    with open(epitope_filename, 'w') as json_file:
        json.dump(new_dict, json_file)
    print(f"Saved epitope JSON file: {epitope_filename}")

print("All epitope JSON files have been saved.")



PART 2
Epitope input: DEFINED USING dSASA -from a known antibody-nanobody complex


Example: HER2-Trastuzumab-Pertuzumab complex (PDB_ID:8PWH) from the Protein DataBase
    Trastuzumab Fab light chain: A
    Trastuzumab Fab heavy chain: B
    Pertuzumab Fab light chain: C
    Pertuzumab Fab heavy chain: D

In this case, run the notebook twice, one per antibody/nanobody molecule bound to the antigen.
Using a nanobody targeting Caplacizumab nanobody as scaffold (PDB_ID:7eow) from the Protein DataBase

In [ ]:
# use nanodesigner1 kernel
import sys
import os
import shutil
import json
import traceback
# sys.path.append("/NanoDesigner")  # Add the NanoDesigner directory to the Python path
# PROJECT_DIR is already on sys.path (see setup cell)

from functionalities.complex_analysis import (
    fetch_from_sabdab, fetch_from_pdb, extract_antibody_info,
    get_cdr_pos
)

from functionalities.nanobody_antibody_interacting_residues import interacting_residues, interacting_residues_extended
from dyMEAN.data.pdb_utils import Protein  
from dyMEAN.utils.renumber import renumber_pdb


In [ ]:
# Define the configuration dictionary
config = {
    "Nano_source_pdb": "7eow",  # Nanobody PDB ID to be used as scaffold
    "nano_chain_id": "B",  # Nanobody heavy chain
    "light_chain_id": "",  # Set to empty if nanobody has no light chain
    "Antigen_complex_source_pdb": "7JIC",  # Antigen PDB ID. Antigen bound to either a nanobody or antibody
    "antigen_chain_id": ["A"],  # List of antigen chain IDs involved in interaction, for now use only one-chain antigens
    "immunoglobulin_heavy_chain": "H",  # List of nanobody/antibody chain IDs involved in interaction to the antigen
    "immunoglobulin_light_chain": "L",  # List of nanobody/antibody chain IDs involved in interaction to the antigen
    "out_dir": WORK_DIR,  # Output directory
    "numbering_scheme": "imgt",  # Numbering scheme to use, e.g., imgt or chothia
    "tmp_dir": os.path.join(WORK_DIR, "temporal_files")  # Temporary directory
}

In [ ]:
# Set the output and temporary directories
out_dir = config["out_dir"]
os.makedirs(out_dir, exist_ok=True)

tmp_dir = config["tmp_dir"]
os.makedirs(tmp_dir, exist_ok=True)

# Define all_structures directory for local look-up
all_structures_dir = ALL_STRUCTURES_DIR
numbering_scheme = config["numbering_scheme"]

# Define paths for local nanobody PDB and temporary download destination
nano_local_path = os.path.join(all_structures_dir, numbering_scheme, config["Nano_source_pdb"] + ".pdb")
nano_source = os.path.join(tmp_dir, config["Nano_source_pdb"] + "_raw.pdb")
antigen_source = os.path.join(tmp_dir, config["Antigen_complex_source_pdb"] + "_raw.pdb")

print("Output directory created:", out_dir)
print("Temporary directory created:", tmp_dir)


In [ ]:
# Download the scaffold nanobody and the antigen complex.
nano_source = os.path.join(tmp_dir, config["Nano_source_pdb"] + "_raw.pdb")
antigen_source = os.path.join(tmp_dir, config["Antigen_complex_source_pdb"] + "_raw.pdb")

print("Scaffold:")
fetch_structure(config["Nano_source_pdb"], nano_source,
                config["numbering_scheme"], all_structures_dir)
print("Antigen complex:")
fetch_structure(config["Antigen_complex_source_pdb"], antigen_source,
                config["numbering_scheme"], all_structures_dir)


In [ ]:
# Define paths for the cleaned-up and renumbered PDB files
nano_renumbered = os.path.join(tmp_dir, config["Nano_source_pdb"] + "_imgt.pdb")
antigen_renumbered = os.path.join(tmp_dir, config["Antigen_complex_source_pdb"] + "_imgt.pdb")

# Renumber nanobody and antigen files using the specified numbering scheme
try:
    print("Renumbering nanobody PDB file...")
    renumber_pdb(nano_source, nano_renumbered, config["numbering_scheme"])
    
    print("Renumbering antigen PDB file...")
    renumber_pdb(antigen_source, antigen_renumbered, config["numbering_scheme"])
    
except Exception as e:
    print(f"Error during renumbering: {e}")

In [ ]:
# Initialize a dictionary to store nanobody information
nanobody_info = {
    "pdb": config["Nano_source_pdb"],
    "heavy_chain": config["nano_chain_id"],
    "light_chain": config["light_chain_id"],
}

numbering = config["numbering_scheme"]

# Load the renumbered nanobody PDB to extract CDR information
try:
    # Load the renumbered nanobody
    pdb_nano = Protein.from_pdb(nano_renumbered, [config["nano_chain_id"]])
    
    # Extract CDR positions using the numbering scheme
    cdr_pos_dict = extract_antibody_info(pdb_nano, config["nano_chain_id"], config["light_chain_id"], config["numbering_scheme"])
    
    # Get the sequences and positions for each CDR
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., 'h1', 'h2', 'h3'
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        
        if cdr_pos:
            nanobody_info[f'cdr{cdr_name}_pos'] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb_nano.peptides.get(config["nano_chain_id"])
            nanobody_info[f'cdr{cdr_name}_seq'] = nano_peptide.get_span(start, end + 1).get_seq()
    
    print("CDR extraction completed.")

    nanobody_info['heavy_chain_seq'] = ""
    for peptide_id, peptide in pdb_nano.peptides.items():
        sequence = peptide.get_seq()
        print(f"Peptide ID: {peptide_id}, Sequence: {sequence}")
        # Append the sequence to the 'antigen_seqs' list
        nanobody_info['heavy_chain_seq'] += sequence


    original_dir = os.path.dirname(nano_renumbered)
    original_filename = os.path.basename(nano_renumbered)
    cleaned_filename = original_filename.replace(f"_{numbering}", "") 
    cleaned_pdb_path = os.path.join(original_dir, cleaned_filename)
    
    pdb_nano.to_pdb(cleaned_pdb_path)
    nanobody_info["nano_source"] = cleaned_pdb_path

    print("heavy chain sequence extracted")
except Exception as e:
    print(f"Error extracting CDRs: {e}")



# add "heavy_chain_seq" key/value 

In [ ]:
# Initialize a dictionary to store nanobody information
temporal_info = {
    "pdb": config["Antigen_complex_source_pdb"],
    "heavy_chain": config["immunoglobulin_heavy_chain"],
    "light_chain": config["immunoglobulin_light_chain"],
    "antigen_chains": config["antigen_chain_id"]
}

H, L, A = config["immunoglobulin_heavy_chain"], config["immunoglobulin_light_chain"], config["antigen_chain_id"]
print(A)
chains_to_iterate = [H, L] if L != "" else [H]
all_complex_chains = chains_to_iterate + A
numbering = config["numbering_scheme"]

# Load the renumbered nanobody PDB to extract CDR information
try:
    # Load the complex
    pdb_complex = Protein.from_pdb(antigen_renumbered, all_complex_chains)
    
    # Extract CDR positions using the numbering scheme
    cdr_pos_dict = extract_antibody_info(pdb_complex, H, L, numbering)
    
    # Get the sequences and positions for each CDR
    for i in range(1, 4):
        cdr_name = f'H{i}'.lower()  # e.g., 'h1', 'h2', 'h3'
        cdr_pos = get_cdr_pos(cdr_pos_dict, f'H{i}')
        
        if cdr_pos:
            temporal_info[f'cdr{cdr_name}_pos'] = cdr_pos
            start, end = cdr_pos
            nano_peptide = pdb_complex.peptides.get(H)
            temporal_info[f'cdr{cdr_name}_seq'] = nano_peptide.get_span(start, end + 1).get_seq()
    
    if L:
        for i in range(1, 4):
            cdr_name = f'L{i}'.lower()  
            cdr_pos = get_cdr_pos(cdr_pos_dict, f'L{i}')
            
            if cdr_pos:
                temporal_info[f'cdr{cdr_name}_pos'] = cdr_pos
                start, end = cdr_pos
                light_peptide = pdb_complex.peptides.get(L)
                temporal_info[f'cdr{cdr_name}_seq'] = light_peptide.get_span(start, end + 1).get_seq()

    print("CDR extraction completed.")

    ag_chains_to_reconstruct = A    
    for antigen_chain in ag_chains_to_reconstruct:
        try:
            pdb = Protein.from_pdb(antigen_renumbered, ag_chains_to_reconstruct)
            temporal_info['antigen_seqs'] = []
        
            for peptide_id, peptide in pdb.peptides.items():
                sequence = peptide.get_seq()
                print(f"Peptide ID: {peptide_id}, Sequence: {sequence}")
                # Append the sequence to the 'antigen_seqs' list
                temporal_info['antigen_seqs'].append(sequence)

        except Exception as e:
            print(f'Something went wrong for {antigen_renumbered}, {e}')
            print(traceback.format_exc())

    # Modify the path to save the cleaned PDB without "_imgt"
    original_dir = os.path.dirname(antigen_renumbered)
    original_filename = os.path.basename(antigen_renumbered)
    cleaned_filename = original_filename.replace(f"_{numbering}", "") 
    cleaned_pdb_path = os.path.join(original_dir, cleaned_filename)

    # Save the cleaned PDB structure to the new path

    pdb.to_pdb(cleaned_pdb_path)
    print(f"Cleaned PDB file saved to: {cleaned_pdb_path}")
    temporal_info["antigen_source"] = cleaned_pdb_path

except Exception as e:
    print(f"Error during processing: {e}")
    print(traceback.format_exc())

In [ ]:
def binding_residues_analysis(temporal_dir, item, complex_pdb_file):
    pdb_file = complex_pdb_file
    heavy_chain = item["heavy_chain"]
    light_chain = item["light_chain"]  # may be an empty string ""
    antigen_chains = item["antigen_chains"]
    pdb_id = item["pdb"]

    # Initialize epitope model to store all interaction results
    epitope_model = []

    # Iterate through each antigen chain to check interactions
    for antigen_chain in antigen_chains:
        for chain in [heavy_chain, light_chain]:
            if not chain:  # Skip if light chain is empty
                continue
            
            chain_type = 'H' if chain == heavy_chain else 'L'
            print(f"Processing antigen {antigen_chain} with {chain_type} chain")

            chains_to_reconstruct = [chain, antigen_chain]
            
            # Generate a unique filename for each antigen-chain pair
            tmp_pdb = os.path.join(temporal_dir, f"{pdb_id}_{chain_type}_chain_{antigen_chain}.pdb")
            if not os.path.exists(tmp_pdb):
                try:
                    # Extract and save PDB file for each chain pair
                    protein = Protein.from_pdb(pdb_file, chains_to_reconstruct)
                    protein.to_pdb(tmp_pdb)
                    assert os.path.exists(tmp_pdb), f"Temporary PDB file not created: {tmp_pdb}"
                except Exception as e:
                    print(f"Failed to process PDB file '{pdb_file}' with {chain_type} chain and antigen {antigen_chain}: {e}")
                    continue

            # Analyze interaction for each antigen-chain pair
            result = interacting_residues_extended(item, tmp_pdb, antigen_chain, temporal_dir, chain_type)
            if result is not None:
                epitope_model.extend(result)  # Collect results if they exist
            
    return epitope_model

In [ ]:
tmp_dir_for_analysis = os.path.join(config["tmp_dir"], "dSASA_computations")
os.makedirs(tmp_dir_for_analysis, exist_ok=True)
epitope = binding_residues_analysis(tmp_dir_for_analysis,temporal_info, antigen_renumbered)


In [ ]:
# Create output directory if it doesn’t exist
output_dir_path = os.path.join(out_dir, f"{config['Nano_source_pdb']}_{config['Antigen_complex_source_pdb']}")
os.makedirs(output_dir_path, exist_ok=True)

# Count existing JSON files in the directory that match the pattern
existing_files = [f for f in os.listdir(output_dir_path) if f.endswith('.json')]
ep = len(existing_files) + 1  # Set ep based on the count of existing JSON files

# Construct the JSON filename
json_file = os.path.join(output_dir_path, f"{config['Nano_source_pdb']}_{config['Antigen_complex_source_pdb']}_ep_{ep}.json")

# Copy nanobody_info to new_item
new_item = nanobody_info.copy()  # Create a shallow copy to modify

# Merge needed information into new_item
nano_scaffold_pdb = nanobody_info["pdb"]
antigen_source_pdb = temporal_info["pdb"]

new_item["pdb"] = f"{nano_scaffold_pdb}_{antigen_source_pdb}_ep_{ep}"
new_item["antigen_chains"] = temporal_info["antigen_chains"]
new_item["antigen_source"] = temporal_info["antigen_source"]

# Move "antigen_source" file to the newly generated output_dir_path and update the key/value
antigen_source_path = os.path.join(config["tmp_dir"], f"{temporal_info['antigen_source']}")
antigen_source_filename = os.path.basename(temporal_info['antigen_source'])
new_antigen_source_path = os.path.join(output_dir_path, antigen_source_filename)
shutil.move(antigen_source_path, new_antigen_source_path)
new_item["antigen_source"] = new_antigen_source_path

# Copy the "nano_source" file to the newly generated output_dir_path and update the key/value
nano_source_path = os.path.join(config["tmp_dir"], f"{config['Nano_source_pdb']}.pdb")
new_nano_source_path = os.path.join(output_dir_path, f"{config['Nano_source_pdb']}.pdb")
shutil.copy(nano_source_path, new_nano_source_path)
new_item["nano_source"] = new_nano_source_path

new_item["antigen_seqs"] = temporal_info["antigen_seqs"]
new_item["epitope_user_input"] = "yes"
new_item["epitope"] = epitope
new_item["numbering"] = config["numbering_scheme"]

# Delete everything in the config["tmp_dir"] directory
for filename in os.listdir(config["tmp_dir"]):
    file_path = os.path.join(config["tmp_dir"], filename)
    try:
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)  # Remove the file or symbolic link
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)  # Remove the directory and all its contents
    except Exception as e:
        print(f"Failed to delete {file_path}. Reason: {e}")

# Save new_item to JSON
with open(json_file, 'w') as f:
    json.dump(new_item, f)

print(f"Saved nanobody information to: {json_file}")
print("Make sure the content of the newly generated json is not repeated")

### Check the generated JSON

Run this before using the file with the pipeline.


In [ ]:
# ============================================================================
#  Validate the generated JSON before running the pipeline
#  Paste this as a new cell at the end of Part 2, after the cell that writes
#  the JSON. Run it on the file that cell reports.
# ============================================================================
import json
import os

# Path printed by the previous cell.
json_to_check = json_file

with open(json_to_check) as fh:
    item = json.load(fh)

problems = []
warnings = []

# --- scaffold ---------------------------------------------------------------
if not item.get("heavy_chain_seq"):
    problems.append(
        "heavy_chain_seq is empty. The scaffold CDR extraction failed. Check that "
        f"chain '{item.get('heavy_chain')}' exists in the renumbered scaffold PDB, and "
        "re-read the output of the nanobody_info cell for an 'Error extracting CDRs' message."
    )

missing_cdrs = [f"cdrh{i}_seq" for i in (1, 2, 3) if not item.get(f"cdrh{i}_seq")]
if missing_cdrs:
    problems.append(
        f"Missing CDR fields: {', '.join(missing_cdrs)}. The pipeline needs all three."
    )

# --- epitope ----------------------------------------------------------------
epitope = item.get("epitope") or []
if not epitope:
    problems.append(
        "epitope is empty. Nothing was detected at the interface. Usual causes:\n"
        "      - dr_sasa is not built, or not at $NANOBODIES_PROJECT_DIR/dr_sasa_n/build/dr_sasa\n"
        "      - the immunoglobulin_heavy_chain / _light_chain ids are wrong for this entry\n"
        "      - the antigen chain does not actually contact those chains"
    )
elif len(epitope) < 5:
    warnings.append(f"Only {len(epitope)} epitope residues - unusually small, worth checking.")
elif len(epitope) > 50:
    warnings.append(f"{len(epitope)} epitope residues - unusually large, check the chain ids.")

# --- antigen ----------------------------------------------------------------
if not item.get("antigen_seqs") or not any(item["antigen_seqs"]):
    problems.append("antigen_seqs is empty. The antigen chain was not extracted.")

# --- structures on disk -----------------------------------------------------
for key in ("nano_source", "antigen_source"):
    path = item.get(key)
    if not path:
        problems.append(f"{key} is not set.")
    elif not os.path.isfile(path):
        problems.append(f"{key} points at a file that does not exist: {path}")

# --- epitope chains must be antigen chains ----------------------------------
if epitope:
    ag = set(item.get("antigen_chains") or [])
    bad = {e[0] for e in epitope if isinstance(e, (list, tuple)) and len(e) == 3} - ag
    if bad:
        problems.append(
            f"Epitope contains residues from chain(s) {sorted(bad)}, which are not in "
            f"antigen_chains {sorted(ag)}. The chain assignment is probably wrong."
        )

# --- report -----------------------------------------------------------------
print(f"Checking {json_to_check}\n")
if problems:
    print("NOT READY - fix these before running the pipeline:\n")
    for p in problems:
        print(f"  [X] {p}")
else:
    print("Looks good:")
    print(f"  scaffold {item['heavy_chain']}  ({len(item['heavy_chain_seq'])} aa)")
    print(f"  CDRs     H1={item.get('cdrh1_seq')}  H2={item.get('cdrh2_seq')}  H3={item.get('cdrh3_seq')}")
    print(f"  antigen  {item['antigen_chains']}  ({sum(len(s) for s in item['antigen_seqs'])} aa)")
    print(f"  epitope  {len(epitope)} residues")

for w in warnings:
    print(f"  [!] {w}")


---

## Optional: build a VHH scaffold structure from its sequence

Use this only if you want to design from a nanobody **sequence** and have no
structure for it. It writes `scaffold.pdb`, which you can then point
`Nano_source_pdb` at in Part 1 or Part 2 instead of a PDB/SAbDab code.

Requires `ImmuneBuilder` (`pip install ImmuneBuilder`).


In [ ]:
import os

from ImmuneBuilder import NanoBodyBuilder2

# 1. Your VHH scaffold sequence (replace with your own).
scaffold_seq = (
    "QVQLQESGGGLVQAGGSLRLSCAASGDSYTAYVMGWFRQAPGKEREFVASITPTNDNFSYYADSVKG"
    "RFTISRDNAKNTVYLQMNSLKPEDTAVYYCALATHSNRPTPITYWGQGTQVTVSS"
)

# 2. Where to write it.
os.makedirs(WORK_DIR, exist_ok=True)
output_pdb = os.path.join(WORK_DIR, "scaffold.pdb")

print("Predicting scaffold structure with ImmuneBuilder (NanoBodyBuilder2)...")
try:
    NanoBodyBuilder2().predict({"H": scaffold_seq}).save(output_pdb)
    print("Saved to:", output_pdb)
    print("Use this file as the nanobody source, with chain id 'H'.")
except Exception as exc:
    print("Structure prediction failed:", exc)
